<a href="https://colab.research.google.com/github/fwitschel/WIMA/blob/main/notebooks/RAG_Hybrid_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Execute this code only if in colab
if 'COLAB_GPU' in os.environ:
  print("Executing in Colab!")
  # Cloning GitHub repository
  !git clone https://github.com/fwitschel/WIMA.git
  %cd WIMA


In [ ]:
!pip install langchain langchain-community rank_bm25 pypdf unstructured chromadb groq
!pip install unstructured['pdf'] unstructured
!pip install nltk
!pip install -U sentence-transformers transformers

In [ ]:
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain.llms import HuggingFaceHub
import torch
from transformers import ( AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, )
from langchain import HuggingFacePipeline
from sentence_transformers import SentenceTransformer

from langchain.retrievers import BM25Retriever, EnsembleRetriever

import os
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

In [ ]:
from langchain.document_loaders import PyPDFLoader
doc_path = "/content/WIMA/data/rueckholung.pdf"
loader = PyPDFLoader(doc_path)
pages = loader.load_and_split()

In [ ]:
from google.colab import userdata
embeddings = HuggingFaceEmbeddings()
vectorstore = Chroma.from_documents(pages, embeddings)

In [ ]:
from nltk.tokenize import word_tokenize

vectorstore_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
keyword_retriever = BM25Retriever.from_documents(pages, preprocess_func=word_tokenize)
keyword_retriever.k =  5

In [ ]:
#query = "Strahlenschutz: Welche speziellen Strahlenschutzmaßnahmen sind während der Rückholung der hochaktiven Abfälle (HAA) vorgesehen, und wie wird die Exposition des Personals minimiert?"
query = "Inwiefern werden durch das Rückholungskonzept die Anforderungen der aktuellen Kernenergieverordnung (KEV) und des Kernenergiegesetzes (KEG) erfüllt? Welche Gesetzesartikel spielen dabei jeweils eine Rolle?"

#results = vectorstore_retriever.invoke(query)
results = keyword_retriever.invoke(query)

for i in range(5):
  print(results[i])

In [ ]:
from groq import Groq
def llm(groq_client, prompt):
  chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="llama-3.3-70b-versatile",
  )

  return chat_completion.choices[0].message.content

In [ ]:
groq_client = Groq(
    api_key=userdata.get('GROQ_API_KEY')
)

In [ ]:
context = '\n\n'.join(list(map(lambda c: "page " + c.metadata["page_label"] + c.page_content, results)))
prompt = f"""<TODO für euch!
        """
response = llm(groq_client, prompt)
print(response)